<a href="https://colab.research.google.com/github/eternitvy/ML_26/blob/main/JS09/JS09_TUGAS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install faiss-cpu annoy hnswlib

In [7]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv('songs_with_attributes_and_lyrics.csv')  # ganti path sesuai lokasi file
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

In [8]:
from sklearn.neighbors import NearestNeighbors

k = 10  # jumlah nearest neighbors
query_vectors = X_scaled[:5] # Mengambil 5 sampel lagu pertama sebagai contoh query

start_time = time.time()
exact_nn = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
exact_nn.fit(X_scaled)

distances_exact, indices_exact = exact_nn.kneighbors(query_vectors)
exact_time = time.time() - start_time

print(f"Waktu Eksekusi Exact NN: {exact_time:.4f} detik")
print("Indeks Tetangga Terdekat (Exact):")
print(indices_exact)

Waktu Eksekusi Exact NN: 2.6415 detik
Indeks Tetangga Terdekat (Exact):
[[     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
 [     1 929001 638076  18923 709894 116786 768496 284535 123477 287426]
 [     2 733400 495761 391522 245380 394744 920542 920543  15252 220937]
 [     3 321694 607119 734643 586500 782644 687571 916328 476573 226078]
 [     4 340543 570674 661781 205475 906778 559999 663572 892945  89941]]


In [9]:
from annoy import AnnoyIndex

f_dim = X_scaled.shape[1]
t = AnnoyIndex(f_dim, 'euclidean')

start_time = time.time()
# Masukkan data ke dalam Annoy index
for i, vector in enumerate(X_scaled):
    t.add_item(i, vector)

# Build index dengan 10 trees
t.build(10)
annoy_build_time = time.time() - start_time

# Query pencarian
start_time = time.time()
indices_annoy = []
distances_annoy = []
for q in query_vectors:
    idx, dist = t.get_nns_by_vector(q, k, include_distances=True)
    indices_annoy.append(idx)
    distances_annoy.append(dist)
annoy_query_time = time.time() - start_time

print(f"ANNOY - Build Time: {annoy_build_time:.4f}s | Query Time: {annoy_query_time:.4f}s")

ANNOY - Build Time: 10.1628s | Query Time: 0.0007s


In [10]:
import faiss

dimension = X_scaled.shape[1]

start_time = time.time()
# Membuat indeks FAISS untuk L2 (Euclidean) distance
index_faiss = faiss.IndexFlatL2(dimension)
index_faiss.add(X_scaled)
faiss_build_time = time.time() - start_time

# Query pencarian
start_time = time.time()
distances_faiss, indices_faiss = index_faiss.search(query_vectors, k)
faiss_query_time = time.time() - start_time

print(f"FAISS - Build Time: {faiss_build_time:.4f}s | Query Time: {faiss_query_time:.4f}s")

FAISS - Build Time: 0.0275s | Query Time: 0.0270s


In [11]:
import hnswlib

dim = X_scaled.shape[1]
num_elements = X_scaled.shape[0]

# Inisialisasi HNSW index
p = hnswlib.Index(space='l2', dim=dim)
p.init_index(max_elements=num_elements, ef_construction=200, M=16)

start_time = time.time()
p.add_items(X_scaled, np.arange(num_elements))
p.set_ef(50) # Parameter kedalaman pencarian saat query
hnsw_build_time = time.time() - start_time

# Query pencarian
start_time = time.time()
indices_hnsw, distances_hnsw = p.knn_query(query_vectors, k=k)
hnsw_query_time = time.time() - start_time

print(f"HNSW - Build Time: {hnsw_build_time:.4f}s | Query Time: {hnsw_query_time:.4f}s")

HNSW - Build Time: 131.1394s | Query Time: 0.0009s


In [12]:
def calculate_recall(exact_idx, approx_idx):
    recalls = []
    for ex, ap in zip(exact_idx, approx_idx):
        intersection = len(set(ex).intersection(set(ap)))
        recalls.append(intersection / len(ex))
    return np.mean(recalls)

print("=== LAPORAN PERBANDINGAN RECALL@10 (terhadap Exact NN) ===")
print(f"ANNOY Recall : {calculate_recall(indices_exact, indices_annoy) * 100:.2f}%")
print(f"FAISS  Recall : {calculate_recall(indices_exact, indices_faiss) * 100:.2f}%")
print(f"HNSW   Recall : {calculate_recall(indices_exact, indices_hnsw) * 100:.2f}%")

=== LAPORAN PERBANDINGAN RECALL@10 (terhadap Exact NN) ===
ANNOY Recall : 86.00%
FAISS  Recall : 100.00%
HNSW   Recall : 100.00%
